# 05 HRV Domain And Robustness

Reviewer-facing HRV-domain and robustness readiness/status notebook.


## Setup


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

# Keep all git operations out of Drive-backed cwd. This avoids Colab
# "Transport endpoint is not connected" failures after Drive hiccups.
os.chdir('/content')
print('cwd reset:', Path.cwd())

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive mount skipped or already mounted: {exc}')

REPO_URL = 'https://github.com/BrianNguyen29/ECG-RAMBA.git'
BRANCH = os.environ.get('ECG_RAMBA_BRANCH', 'main')
DRIVE_ROOT = Path('/content/drive/MyDrive/ECG-Ramba')
MIRROR_REVISION_ROOT = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
LOCAL_RUNTIME_ROOT = Path('/content/ecg_ramba_runtime')
REPO_DIR = Path('/content/ECG-RAMBA')

os.environ['ECG_RAMBA_DRIVE_ROOT'] = str(DRIVE_ROOT)
os.environ.setdefault('ECG_RAMBA_LOCAL_ROOT', str(LOCAL_RUNTIME_ROOT))
os.environ.setdefault('ECG_RAMBA_EXTRACT_DIR', str(LOCAL_RUNTIME_ROOT / 'chapman'))
os.environ.setdefault('ECG_RAMBA_USE_CLEAN_CACHE', '0')
os.environ.setdefault('ECG_RAMBA_SAVE_CLEAN_CACHE', '0')

def _run_setup(cmd, cwd='/content', check=True):
    print(f'$ {cmd}', flush=True)
    result = subprocess.run(
        cmd,
        shell=True,
        cwd=str(cwd) if cwd else None,
        check=False,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.stdout:
        print(result.stdout.rstrip())
    if check and result.returncode:
        raise subprocess.CalledProcessError(result.returncode, cmd, output=result.stdout)
    return result

def _is_repo(path):
    path = Path(path)
    return (path / '.git').exists() and (path / 'configs' / 'config.py').exists()

def _remove_local_repo(reason):
    if REPO_DIR.exists():
        print(f'Removing local repo at {REPO_DIR}: {reason}')
        _run_setup(f'rm -rf "{REPO_DIR}"', cwd='/content')

if REPO_DIR.exists() and not _is_repo(REPO_DIR):
    _remove_local_repo('path exists but is not a valid repo')

if not REPO_DIR.exists():
    _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', cwd='/content')
else:
    print('Using existing local repo:', REPO_DIR)

lock_path = REPO_DIR / '.git' / 'index.lock'
if lock_path.exists():
    print('Removing stale git index lock:', lock_path)
    lock_path.unlink()

fetch_result = _run_setup('git fetch origin', cwd=REPO_DIR, check=False)
checkout_result = _run_setup(f'git checkout {BRANCH}', cwd=REPO_DIR, check=False)
if fetch_result.returncode == 0 and checkout_result.returncode == 0:
    pull_result = _run_setup(f'git pull --ff-only --autostash origin {BRANCH}', cwd=REPO_DIR, check=False)
    if pull_result.returncode:
        print('git pull failed; replacing /content clone with a fresh checkout.')
        _remove_local_repo('pull failed')
        _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', cwd='/content')
else:
    print('git fetch/checkout failed; replacing /content clone with a fresh checkout.')
    _remove_local_repo('fetch or checkout failed')
    _run_setup(f'git clone --branch {BRANCH} {REPO_URL} "{REPO_DIR}"', cwd='/content')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('cwd       :', Path.cwd())
print('drive_root:', DRIVE_ROOT)
print('repo_dir  :', REPO_DIR)
print('branch    :', BRANCH)
_run_setup('git rev-parse --short HEAD', cwd=REPO_DIR, check=False)
_run_setup('git status --short --branch', cwd=REPO_DIR, check=False)


DRIVE_REPO_REVISION_ROOT = DRIVE_ROOT / 'ECG-RAMBA' / 'reports' / 'revision'
REQUIRED_REVISION_ARTIFACTS_FOR_05 = [
    'manifests/oof_final_ema_freeze_manifest.json',
    'predictions/oof_final_ema_predictions.npz',
    'metrics/calibration_ci_oof_final_ema_predictions.json',
    'metrics/baseline_summary.csv',
    'metrics/component_check_summary.json',
    'audit_protocol.json',
    'a0_resolution_status.json',
    'hrv36_schema.csv',
]


def _sha256_local(path):
    import hashlib
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _mirror_manifest_rows(mirror_root):
    manifest_path = Path(mirror_root) / 'manifests' / 'mirror_manifest.json'
    if not manifest_path.exists():
        return manifest_path, {}
    payload = json.loads(manifest_path.read_text(encoding='utf-8'))
    declared_root = Path(payload.get('mirror_root', mirror_root))
    rows = {}
    for row in payload.get('artifacts', []):
        relative = row.get('relative_path')
        if not relative and row.get('mirror'):
            mirror_path = Path(row['mirror'])
            try:
                relative = mirror_path.relative_to(declared_root).as_posix()
            except ValueError:
                relative = mirror_path.name
        if relative:
            rows[Path(relative).as_posix()] = row
    return manifest_path, rows


def restore_required_revision_artifacts(required_rel_paths):
    import shutil

    destination_root = REPO_DIR / 'reports' / 'revision'
    manifest_path, rows = _mirror_manifest_rows(MIRROR_REVISION_ROOT)
    restored, mirror_path_copied, fallback_copied, reused, unresolved = [], [], [], [], []
    for rel in required_rel_paths:
        rel = Path(rel).as_posix()
        destination = destination_root / rel
        row = rows.get(rel)
        needs_copy = not destination.exists()
        if row and destination.exists() and _sha256_local(destination) != row.get('sha256'):
            needs_copy = True
        if not needs_copy:
            reused.append(rel)
            continue

        source = MIRROR_REVISION_ROOT / rel
        copied = False
        if row and source.exists():
            expected_size = int(row.get('size_bytes', -1))
            if expected_size >= 0 and source.stat().st_size != expected_size:
                raise RuntimeError(f'Mirror size mismatch for required artifact: {rel}')
            if _sha256_local(source) != row.get('sha256'):
                raise RuntimeError(f'Mirror checksum mismatch for required artifact: {rel}')
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            if _sha256_local(destination) != row.get('sha256'):
                raise RuntimeError(f'Checksum mismatch after restoring required artifact: {rel}')
            restored.append(rel)
            copied = True

        if not copied and source.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            mirror_path_copied.append(rel)
            copied = True
            print(f'WARNING: copied {rel} from mirror path without a manifest row; downstream SHA/protocol checks still apply.')

        fallback_source = DRIVE_REPO_REVISION_ROOT / rel
        if not copied and fallback_source.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(fallback_source, destination)
            fallback_copied.append(rel)
            copied = True
            print(f'WARNING: copied {rel} from Drive repo fallback because verified mirror copy was unavailable.')

        if not copied:
            unresolved.append(rel)

    print(
        'Required revision artifact restore: '
        f'restored={len(restored)} mirror_path={len(mirror_path_copied)} fallback={len(fallback_copied)} reused={len(reused)} unresolved={len(unresolved)}'
    )
    if restored:
        print('Restored required artifacts from verified mirror:')
        for rel in restored:
            print(' -', rel)
    if mirror_path_copied:
        print('Copied required artifacts from mirror path without manifest rows; downstream SHA/protocol checks still apply:')
        for rel in mirror_path_copied:
            print(' -', rel)
    if fallback_copied:
        print('Fallback-copied required artifacts from Drive repo; downstream SHA/protocol checks still apply:')
        for rel in fallback_copied:
            print(' -', rel)
    if unresolved:
        print('WARNING: required artifacts remain missing after mirror/fallback restore. Contract cell will fail if unresolved:')
        for rel in unresolved:
            print(' -', rel)


if MIRROR_REVISION_ROOT.exists():
    restore_result = _run_setup(
        f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
        cwd=REPO_DIR,
        check=False,
    )
    if restore_result.returncode:
        print('WARNING: full checksum-verified mirror restore failed; trying selective restore for required inputs only.')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_05)
else:
    print('Mirror not found; trying Drive repo fallback for required artifacts:', MIRROR_REVISION_ROOT)
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_05)

required_code = [
    'scripts/revision/09_hrv_domain_analysis.py',
    'scripts/revision/12_robustness_stress.py',
    'notebooks/05_hrv_domain_and_robustness.ipynb',
    'docs/revision_plan/experiment_registry.csv',
    'docs/revision_plan/task_board.csv',
]
missing_code = []
for item in required_code:
    file_path = REPO_DIR / item
    status = 'FOUND' if file_path.exists() else 'MISSING'
    size = file_path.stat().st_size if file_path.exists() else None
    print(f'{item}: {status} size={size}')
    if not file_path.exists():
        missing_code.append(item)
if missing_code:
    raise FileNotFoundError('Missing required code files after setup: ' + '; '.join(missing_code))

cache_status = {
    'clean_raw_cache': DRIVE_ROOT / 'ecg_data_27c_subject.npz',
    'raw_minirocket_cache': DRIVE_ROOT / 'rocket_raw_N44186_C12_L5000_K10000_S42.npz',
    'hrv36_cache': DRIVE_ROOT / 'hrv36_N44186_C12_L5000.npz',
    'fold_pca_cache_dir': DRIVE_ROOT / 'revision_feature_cache',
    'model_dir_drive': DRIVE_ROOT / 'model',
}
print('cache policy:')
print('  ECG_RAMBA_USE_CLEAN_CACHE =', os.environ.get('ECG_RAMBA_USE_CLEAN_CACHE'))
print('  ECG_RAMBA_SAVE_CLEAN_CACHE=', os.environ.get('ECG_RAMBA_SAVE_CLEAN_CACHE'))
print('  ECG_RAMBA_EXTRACT_DIR     =', os.environ.get('ECG_RAMBA_EXTRACT_DIR'))
print('cache status:')
for name, cache_path in cache_status.items():
    if cache_path.is_dir():
        npz_count = len(list(cache_path.glob('*.npz')))
        pt_count = len(list(cache_path.glob('*.pt')))
        print(f'  {name}: exists=True npz_count={npz_count} pt_count={pt_count} path={cache_path}')
    else:
        print(f'  {name}: exists={cache_path.exists()} size={cache_path.stat().st_size if cache_path.exists() else None} path={cache_path}')

def run(cmd, check=True, log_path=None, tail_lines=160, cwd=None):
    print(f'$ {cmd}', flush=True)
    run_cwd = Path(cwd) if cwd else Path.cwd()
    if log_path is None:
        log_dir = run_cwd / 'reports' / 'revision' / 'logs'
        log_dir.mkdir(parents=True, exist_ok=True)
        stamp = time.strftime('%Y%m%d_%H%M%S')
        log_path = log_dir / f'notebook_command_{stamp}.log'
    else:
        log_path = Path(log_path)
        if not log_path.is_absolute():
            log_path = run_cwd / log_path
        log_path.parent.mkdir(parents=True, exist_ok=True)

    with log_path.open('w', encoding='utf-8') as log_file:
        proc = subprocess.Popen(
            cmd,
            shell=True,
            cwd=str(run_cwd),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding='utf-8',
            errors='replace',
            bufsize=1,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = proc.wait()

    print(f'Command log: {log_path}')
    if check and return_code:
        log_lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
        print(f'Command failed with exit code {return_code}. Last {min(tail_lines, len(log_lines))} log lines:')
        for line in log_lines[-tail_lines:]:
            print(line)
        raise subprocess.CalledProcessError(return_code, cmd)
    return subprocess.CompletedProcess(cmd, return_code)


def run_script_if_exists(script_path, command):
    script = Path(script_path)
    if script.exists():
        run(command)
    else:
        print(f'SKIP: {script_path} is not implemented yet.')
        print(f'Planned command: {command}')

print('Setup ready. Continue with Scope And Contracts cell.')


## Scope And Contracts

This notebook executes HRV-only/domain evidence, Full-vs-MiniRocket perturbation robustness, optional comparator stressed-prediction generation for ResNet1D/CNN and Raw Mamba, and a low-memory multi-comparator robustness ledger. HRV/domain cells do not load neural checkpoints. Robustness inference is deliberately gated behind explicit flags and requires cached predictions or saved comparator checkpoints.


In [ ]:
from datetime import datetime, timezone
import math
import numpy as np
import pandas as pd
from IPython.display import display

from scripts.revision.common import git_commit, save_json, sha256_file

RUN_HRV_DOMAIN_ANALYSIS = True
FORCE_RERUN_HRV_DOMAIN_ANALYSIS = False
RUN_ROBUSTNESS_STRESS = True
FORCE_RERUN_ROBUSTNESS_STRESS = False

revision_root = Path('reports/revision')
OOF_STEM = 'oof_final_ema'
freeze_path = revision_root / 'manifests' / f'{OOF_STEM}_freeze_manifest.json'
oof_prediction_path = revision_root / 'predictions' / f'{OOF_STEM}_predictions.npz'
calibration_path = revision_root / 'metrics' / f'calibration_ci_{OOF_STEM}_predictions.json'
baseline_summary_path = revision_root / 'metrics' / 'baseline_summary.csv'
component_summary_path = revision_root / 'metrics' / 'component_check_summary.json'
audit_path = revision_root / 'audit_protocol.json'
a0_status_path = revision_root / 'a0_resolution_status.json'
hrv_schema_path = revision_root / 'hrv36_schema.csv'

required_inputs = {
    'oof_final_ema_freeze_manifest': freeze_path,
    'oof_final_ema_predictions': oof_prediction_path,
    'calibration_ci': calibration_path,
    'baseline_summary': baseline_summary_path,
    'component_check_summary': component_summary_path,
    'audit_protocol': audit_path,
    'a0_resolution_status': a0_status_path,
    'hrv36_schema': hrv_schema_path,
}
for name, path in required_inputs.items():
    print(f'{name:26s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')
missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing and 'restore_required_revision_artifacts' in globals():
    print('Required inputs missing before contract validation; retrying selective artifact restore...')
    restore_required_revision_artifacts(REQUIRED_REVISION_ARTIFACTS_FOR_05)
    missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing and 'restore_required_revision_artifacts' not in globals():
    print('Setup restore helper is unavailable; attempting direct fallback restore from Drive roots.')
    import shutil
    drive_root = globals().get('DRIVE_ROOT', Path('/content/drive/MyDrive/ECG-Ramba'))
    repo_dir = globals().get('REPO_DIR', Path.cwd())
    mirror_root = drive_root / 'revision_artifacts' / 'reports' / 'revision'
    drive_repo_root = drive_root / 'ECG-RAMBA' / 'reports' / 'revision'
    destination_root = repo_dir / 'reports' / 'revision'
    required_rel_paths = {path.relative_to(revision_root).as_posix(): path for path in required_inputs.values()}
    restored_direct, unresolved_direct = [], []
    for rel, destination in required_rel_paths.items():
        if destination.exists():
            continue
        source = mirror_root / rel
        if not source.exists():
            source = drive_repo_root / rel
        if source.exists():
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            restored_direct.append(rel)
        else:
            unresolved_direct.append(rel)
    print(f'Direct fallback restore: restored={len(restored_direct)} unresolved={len(unresolved_direct)}')
    for rel in restored_direct:
        print(' - restored', rel)
    for rel in unresolved_direct:
        print(' - unresolved', rel)
    missing = [str(path) for path in required_inputs.values() if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Notebook 01/02/03/04 artifacts are required before notebook 05: ' + '; '.join(missing) +
        '. Run Notebook 05 from the Setup cell, and verify these files exist under either '
        '/content/drive/MyDrive/ECG-Ramba/revision_artifacts/reports/revision or '
        '/content/drive/MyDrive/ECG-Ramba/ECG-RAMBA/reports/revision.'
    )

freeze = json.loads(freeze_path.read_text(encoding='utf-8'))
if freeze.get('status') != 'frozen' or freeze.get('manuscript_ready') is not True:
    raise ValueError('final_ema OOF freeze manifest must be status=frozen and manuscript_ready=true before HRV/robustness review.')
if freeze.get('checkpoint_kind') != 'final_ema':
    raise ValueError('Notebook 05 requires checkpoint_kind=final_ema in the freeze manifest.')
frozen_artifacts = {row['path']: row for row in freeze.get('artifacts', [])}
relative_oof_predictions = oof_prediction_path.as_posix()
if relative_oof_predictions not in frozen_artifacts:
    raise ValueError(f'Freeze manifest does not include {relative_oof_predictions}')
oof_prediction_sha = sha256_file(oof_prediction_path)
if oof_prediction_sha != frozen_artifacts[relative_oof_predictions]['sha256']:
    raise RuntimeError('OOF prediction SHA256 changed after final_ema freeze.')

baseline_df_for_contract = pd.read_csv(baseline_summary_path)
full_rows = baseline_df_for_contract[baseline_df_for_contract['baseline_name'] == 'Full ECG-RAMBA frozen OOF']
if full_rows.empty:
    raise RuntimeError('baseline_summary.csv lacks the Full ECG-RAMBA frozen OOF row from Notebook 04.')
if set(full_rows['evidence_path'].astype(str)) != {str(calibration_path)}:
    raise RuntimeError('baseline_summary.csv full-model row is not tied to the canonical final_ema calibration artifact.')
component_summary = json.loads(component_summary_path.read_text(encoding='utf-8'))
if component_summary.get('source_freeze_manifest') != str(freeze_path):
    raise RuntimeError('component_check_summary.json is not tied to the canonical final_ema freeze manifest.')
if component_summary.get('source_calibration') != str(calibration_path):
    raise RuntimeError('component_check_summary.json is not tied to the canonical final_ema calibration artifact.')

input_contract = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'inputs': {
        name: {'path': str(path), 'sha256': sha256_file(path), 'size_bytes': path.stat().st_size}
        for name, path in required_inputs.items()
    },
    'freeze_status': freeze.get('status'),
    'freeze_manuscript_ready': freeze.get('manuscript_ready'),
    'freeze_checkpoint_kind': freeze.get('checkpoint_kind'),
    'oof_predictions_sha256': oof_prediction_sha,
    'allowed_execution': {
        'hrv_only_feature_baseline_training': RUN_HRV_DOMAIN_ANALYSIS,
        'hrv_domain_classifier_training': RUN_HRV_DOMAIN_ANALYSIS,
        'force_rerun_hrv_domain_analysis': FORCE_RERUN_HRV_DOMAIN_ANALYSIS,
        'robustness_stress_runner': RUN_ROBUSTNESS_STRESS,
        'force_rerun_robustness_stress': FORCE_RERUN_ROBUSTNESS_STRESS,
        'full_model_training': False,
        'full_model_inference': RUN_ROBUSTNESS_STRESS,
        'signal_perturbation': RUN_ROBUSTNESS_STRESS,
    },
}
save_json(revision_root / 'manifests' / 'hrv_robustness_input_contract.json', input_contract)
print('Input contract validated and written:', revision_root / 'manifests' / 'hrv_robustness_input_contract.json')


## Revision Plan Alignment

This cell binds notebook 05 to HRV-domain and robustness tasks in the tracked revision plan.


In [ ]:
registry = pd.read_csv('docs/revision_plan/experiment_registry.csv')
claims = pd.read_csv('docs/revision_plan/claim_evidence_map.csv')
tasks = pd.read_csv('docs/revision_plan/task_board.csv')

relevant_experiments = registry[registry['experiment_id'].isin(['EXP_HRV', 'EXP_ROBUST'])]
relevant_claims = claims[claims['claim_id'].isin(['C03'])]
relevant_tasks = tasks[tasks['id'].isin(['A6', 'B2'])]

display(relevant_experiments)
display(relevant_claims)
display(relevant_tasks)

plan_alignment = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'experiments': relevant_experiments.to_dict(orient='records'),
    'claims': relevant_claims.to_dict(orient='records'),
    'tasks': relevant_tasks.to_dict(orient='records'),
}
save_json(revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json', plan_alignment)
print('Wrote:', revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json')


## HRV Domain Evidence

This cell runs the implemented HRV-only baseline and HRV-domain classifier. The HRV-only baseline uses HRV36 features with the same frozen Chapman OOF folds. The domain classifier uses bounded HRV36 feature caches only; it does not open raw signals or run the neural model.


In [ ]:
hrv_schema = pd.read_csv(hrv_schema_path)
print('HRV schema rows:', len(hrv_schema))
display(hrv_schema.head(10))

with np.load(oof_prediction_path, allow_pickle=False) as oof_npz:
    if 'dataset_record_order_fingerprint' not in oof_npz.files:
        raise RuntimeError('Canonical final_ema OOF predictions must contain dataset_record_order_fingerprint.')
    dataset_record_order_fingerprint = str(oof_npz['dataset_record_order_fingerprint'].item())

chapman_hrv_cache = DRIVE_ROOT / f'hrv36_N44186_C12_L5000_R{dataset_record_order_fingerprint}.npz'
if not chapman_hrv_cache.exists():
    available_caches = sorted(path.name for path in DRIVE_ROOT.glob('hrv36_N44186_C12_L5000_R*.npz'))
    raise FileNotFoundError(
        'Missing Chapman HRV cache that matches the final_ema OOF record-order fingerprint: '
        f'{chapman_hrv_cache}. Available fingerprinted caches: {available_caches}. '
        'Run Notebook 02a/train cache generation for the same Chapman record order; do not use a mismatched HRV cache.'
    )
chapman_hrv_cache_sha = sha256_file(chapman_hrv_cache)
print('OOF dataset record-order fingerprint:', dataset_record_order_fingerprint)
print('Using matching Chapman HRV cache:', chapman_hrv_cache)
print('Chapman HRV cache sha256:', chapman_hrv_cache_sha)

hrv_command = (
    'python -u scripts/revision/09_hrv_domain_analysis.py '
    f'--oof-predictions {oof_prediction_path} '
    f'--freeze-manifest {freeze_path} --expected-checkpoint-kind final_ema '
    f'--chapman-hrv-cache "{chapman_hrv_cache}" '
    '--threshold 0.5 --n-bins 15 --n-boot 1000 --domain-max-per-domain 2000'
)
hrv_required_outputs = [
    revision_root / 'predictions' / 'hrv_only_oof_predictions.npz',
    revision_root / 'metrics' / 'hrv_only_baseline_summary.json',
    revision_root / 'metrics' / 'hrv_domain_classifier_summary.json',
    revision_root / 'metrics' / 'hrv_domain_summary.csv',
    revision_root / 'tables' / 'table_hrv_only_class_metrics.csv',
    revision_root / 'tables' / 'table_hrv_domain_status.csv',
    revision_root / 'manifests' / 'hrv_domain_analysis_manifest.json',
]
hrv_optional_outputs = [
    revision_root / 'predictions' / 'hrv_domain_oof_predictions.npz',
    revision_root / 'tables' / 'table_hrv_domain_classifier_confusion.csv',
    revision_root / 'tables' / 'table_hrv_domain_classifier_fold_summary.csv',
]
hrv_artifacts_ready = all(output.exists() and output.stat().st_size > 0 for output in hrv_required_outputs)
if RUN_HRV_DOMAIN_ANALYSIS and (FORCE_RERUN_HRV_DOMAIN_ANALYSIS or not hrv_artifacts_ready):
    run(hrv_command, log_path='reports/revision/logs/hrv_domain_analysis.log')
elif hrv_artifacts_ready:
    print('Reusing existing HRV/domain artifacts. Set FORCE_RERUN_HRV_DOMAIN_ANALYSIS=True to recompute.')
else:
    raise RuntimeError('RUN_HRV_DOMAIN_ANALYSIS is False and required HRV/domain artifacts are missing; HRV evidence will remain blocked.')

for output in hrv_required_outputs:
    print(output, 'required exists=', output.exists(), 'size=', output.stat().st_size if output.exists() else None)
missing_hrv = [str(output) for output in hrv_required_outputs if not output.exists()]
if missing_hrv:
    raise FileNotFoundError('Missing required HRV analysis outputs: ' + '; '.join(missing_hrv))
for output in hrv_optional_outputs:
    print(output, 'optional exists=', output.exists(), 'size=', output.stat().st_size if output.exists() else None)

hrv_df = pd.read_csv(revision_root / 'metrics' / 'hrv_domain_summary.csv')
hrv_baseline_summary = json.loads((revision_root / 'metrics' / 'hrv_only_baseline_summary.json').read_text(encoding='utf-8'))
hrv_domain_summary = json.loads((revision_root / 'metrics' / 'hrv_domain_classifier_summary.json').read_text(encoding='utf-8'))
hrv_load_info = hrv_baseline_summary.get('load_info', {})
hrv_freeze_contract = hrv_load_info.get('freeze_contract', {}) or {}
expected_records = int(freeze.get('validated_records', -1))
expected_fold_counts = {str(k): int(v) for k, v in (freeze.get('fold_counts') or {}).items()}
observed_fold_counts = {str(k): int(v) for k, v in (hrv_load_info.get('fold_counts') or hrv_freeze_contract.get('fold_counts') or {}).items()}
exact_oof_sha_match = hrv_load_info.get('oof_predictions_sha256') == oof_prediction_sha
protocol_compatible_oof = (
    int(hrv_load_info.get('oof_records_used', -1)) == expected_records
    and int(hrv_load_info.get('oof_records_total', expected_records)) == expected_records
    and (not expected_fold_counts or observed_fold_counts == expected_fold_counts)
    and hrv_freeze_contract.get('checkpoint_kind') == 'final_ema'
    and hrv_freeze_contract.get('dataset_record_order_fingerprint', dataset_record_order_fingerprint) == dataset_record_order_fingerprint
)
if not (exact_oof_sha_match or protocol_compatible_oof):
    raise RuntimeError(
        'HRV-only summary is not tied to the canonical final_ema OOF protocol. '
        f"summary_oof_sha={hrv_load_info.get('oof_predictions_sha256')} current_oof_sha={oof_prediction_sha}; "
        f"oof_records_used={hrv_load_info.get('oof_records_used')} expected_records={expected_records}; "
        f"fold_counts={observed_fold_counts} expected_fold_counts={expected_fold_counts}; "
        f"checkpoint_kind={hrv_freeze_contract.get('checkpoint_kind')}"
    )
if not exact_oof_sha_match:
    print(
        'HRV-only summary OOF file SHA differs from the current rewritten OOF NPZ, '
        'but the stable record/fold/final_ema protocol contract matches; reusing HRV artifact.'
    )
if hrv_load_info.get('chapman_hrv_cache_kind') != 'record_fingerprinted':
    raise RuntimeError('HRV-only summary did not use a record-fingerprinted Chapman HRV cache.')
if hrv_load_info.get('chapman_hrv_cache_sha256') != chapman_hrv_cache_sha:
    raise RuntimeError('HRV-only summary cache SHA does not match the selected fingerprinted HRV cache.')
if dataset_record_order_fingerprint not in Path(hrv_load_info.get('chapman_hrv_cache', '')).name:
    raise RuntimeError('HRV-only summary cache does not match the canonical final_ema record-order fingerprint.')
if hrv_freeze_contract.get('checkpoint_kind') != 'final_ema':
    raise RuntimeError('HRV-only summary is not tied to final_ema freeze contract.')

# Publish immediately so a later Notebook 04 rerun can restore the canonical HRV artifacts
# even if the Colab runtime disconnects before the final inventory cell.
mirror_root = globals().get('MIRROR_REVISION_ROOT', DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision')
run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/hrv_domain_immediate_artifact_inventory.log',
)
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/hrv_domain_immediate_mirror_publish.log',
)

print('HRV-only metrics:', json.dumps(hrv_baseline_summary['metrics'], indent=2, sort_keys=True))
print('HRV domain status:', hrv_domain_summary.get('status', 'complete'))
print('HRV domain metrics:', json.dumps(hrv_domain_summary.get('metrics') or {}, indent=2, sort_keys=True))
print('HRV domain interpretation:', hrv_domain_summary.get('interpretation'))
if hrv_domain_summary.get('blocker'):
    print('HRV domain blocker:', hrv_domain_summary.get('blocker'))
display(hrv_df)


## Robustness Stress-Test Ledger

The revised plan requires perturbation tests. This cell is configured as a final aggregation pass: it checks that all six per-stress Full/MiniRocket prediction artifacts exist, restores the Drive mirror once if needed, then runs the robustness script with `--reuse-existing` and Drive-backed metric-level resume caches to regenerate the combined tables, bootstrap CIs, and manifest. If Colab disconnects mid-bootstrap, rerun this cell; completed stress/metric caches are reused instead of recomputed. If a prediction artifact is missing, run that stress individually first and publish the mirror before returning to the full list.


In [ ]:
ROBUSTNESS_STRESS_TESTS = 'snr20db,snr10db,snr5db,random_3_lead_dropout,precordial_dropout,resample_250hz'
ROBUSTNESS_LIMIT_RECORDS = 0
ROBUSTNESS_N_BOOT = 1000
ROBUSTNESS_BATCH_SIZE = int(os.environ.get('ECG_RAMBA_ROBUSTNESS_BATCH_SIZE', '64'))
ROBUSTNESS_NUM_WORKERS = 0
ROBUSTNESS_MINIROCKET_FEATURE_DEVICE = 'cpu'
ROBUSTNESS_ALLOW_TF32 = True
ROBUSTNESS_FINAL_AGGREGATION_PASS = True
ROBUSTNESS_REQUIRE_EXISTING_STRESS_PREDICTIONS = True
ROBUSTNESS_METRIC_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' / 'robustness_metric_cache'

robustness_outputs = {
    'summary_csv': revision_root / 'metrics' / 'robustness_summary.csv',
    'table_csv': revision_root / 'tables' / 'table_robustness.csv',
    'comparison_json': revision_root / 'metrics' / 'robustness_full_vs_minirocket_comparison.json',
    'bootstrap_samples': revision_root / 'metrics' / 'robustness_full_vs_minirocket_bootstrap_samples.csv',
    'manifest': revision_root / 'manifests' / 'robustness_stress_manifest.json',
    'stress_metadata': revision_root / 'tables' / 'table_robustness_stress_metadata.csv',
    'artifacts': revision_root / 'tables' / 'table_robustness_artifacts.csv',
}
robustness_outputs_ready = all(path.exists() and path.stat().st_size > 0 for path in robustness_outputs.values())

robustness_stress_list = [s.strip() for s in ROBUSTNESS_STRESS_TESTS.split(',') if s.strip()]
expected_full_robustness_stresses = [
    'snr20db',
    'snr10db',
    'snr5db',
    'random_3_lead_dropout',
    'precordial_dropout',
    'resample_250hz',
]
expected_robustness_metrics = {'pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro'}
robustness_reference_outputs = [
    revision_root / 'predictions' / 'robustness_minirocket_clean_ref_predictions.npz',
    revision_root / 'manifests' / 'robustness_minirocket_heads_manifest.json',
]

if ROBUSTNESS_FINAL_AGGREGATION_PASS and robustness_stress_list != expected_full_robustness_stresses:
    raise ValueError(
        'Final aggregation pass must use the canonical full stress list. '
        f'Got: {robustness_stress_list}'
    )


def robustness_prediction_status(stress_names):
    rows = []
    for stress_name in stress_names:
        for model_label in ['full', 'minirocket']:
            path = revision_root / 'predictions' / f'robustness_{model_label}_{stress_name}_predictions.npz'
            rows.append({
                'stress_test': stress_name,
                'model': model_label,
                'path': path.as_posix(),
                'exists': path.exists(),
                'size_bytes': path.stat().st_size if path.exists() else None,
            })
    return pd.DataFrame(rows)


def missing_robustness_prediction_paths(status_df):
    return status_df.loc[~status_df['exists'], 'path'].astype(str).tolist()


robustness_predictions_complete = False

if RUN_ROBUSTNESS_STRESS and ROBUSTNESS_FINAL_AGGREGATION_PASS and ROBUSTNESS_REQUIRE_EXISTING_STRESS_PREDICTIONS:
    print('Final aggregation preflight: checking per-stress prediction artifacts before running robustness summary generation.')
    prediction_status = robustness_prediction_status(robustness_stress_list)
    display(prediction_status)
    reference_status = pd.DataFrame([
        {
            'artifact': path.name,
            'path': path.as_posix(),
            'exists': path.exists(),
            'size_bytes': path.stat().st_size if path.exists() else None,
        }
        for path in robustness_reference_outputs
    ])
    display(reference_status)
    missing_predictions = missing_robustness_prediction_paths(prediction_status)
    missing_references = reference_status.loc[~reference_status['exists'], 'path'].astype(str).tolist()
    missing_predictions.extend(missing_references)
    if missing_predictions and 'MIRROR_REVISION_ROOT' in globals() and MIRROR_REVISION_ROOT.exists():
        print('Missing prediction artifacts after setup; attempting one more verified mirror restore before failing.')
        run(
            f'python -u scripts/revision/artifact_mirror.py restore --mirror-root "{MIRROR_REVISION_ROOT}" --replace-mismatched',
            log_path='reports/revision/logs/robustness_final_aggregation_restore.log',
        )
        prediction_status = robustness_prediction_status(robustness_stress_list)
        display(prediction_status)
        reference_status = pd.DataFrame([
            {
                'artifact': path.name,
                'path': path.as_posix(),
                'exists': path.exists(),
                'size_bytes': path.stat().st_size if path.exists() else None,
            }
            for path in robustness_reference_outputs
        ])
        display(reference_status)
        missing_predictions = missing_robustness_prediction_paths(prediction_status)
        missing_predictions.extend(reference_status.loc[~reference_status['exists'], 'path'].astype(str).tolist())
    if missing_predictions:
        raise FileNotFoundError(
            'Final robustness aggregation is configured to reuse completed stress predictions, but these artifacts are missing: '
            + '; '.join(missing_predictions)
            + '. Run the missing stress test(s) individually and publish the mirror, or set '
            'ROBUSTNESS_FINAL_AGGREGATION_PASS=False only if you intentionally want this cell to recompute missing predictions.'
        )
    robustness_predictions_complete = True
    print('All per-stress prediction artifacts are present. The runner should reuse predictions and regenerate the combined tables/manifest.')
    print('Skipping Mamba runtime installation for this final aggregation pass because no Full-model inference should be required.')
    print('Robustness metric cache dir:', ROBUSTNESS_METRIC_CACHE_DIR)

robustness_command = (
    'python -u scripts/revision/12_robustness_stress.py '
    f'--stress-tests {ROBUSTNESS_STRESS_TESTS} '
    f'--limit-records {ROBUSTNESS_LIMIT_RECORDS} '
    f'--checkpoint-kind final_ema --expected-checkpoint-kind final_ema '
    '--oof-run-manifest reports/revision/manifests/oof_final_ema_prediction_run_manifest.json '
    f'--threshold 0.5 --n-bins 15 --n-boot {ROBUSTNESS_N_BOOT} '
    f'--metric-cache-dir "{ROBUSTNESS_METRIC_CACHE_DIR}" '
    f'--batch-size {ROBUSTNESS_BATCH_SIZE} --num-workers {ROBUSTNESS_NUM_WORKERS} '
    f'--minirocket-feature-device {ROBUSTNESS_MINIROCKET_FEATURE_DEVICE} '
    '--minirocket-device auto --minirocket-epochs 20 '
    '--reuse-existing --require-existing-stress-predictions --reuse-metric-cache '
    '--bootstrap-progress-every 100 --reuse-minirocket-heads --save-perturbed-caches'
)
if ROBUSTNESS_ALLOW_TF32:
    robustness_command += ' --allow-tf32'

if RUN_ROBUSTNESS_STRESS and not robustness_predictions_complete:
    import importlib
    import json
    missing_runtime_modules = []
    for module_name in ['mamba_ssm', 'causal_conv1d']:
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            missing_runtime_modules.append((module_name, repr(exc)))
    if missing_runtime_modules:
        print('Mamba runtime modules are missing; running canonical installer from Notebook 02:', missing_runtime_modules)
        installer_nb_path = REPO_DIR / 'notebooks' / '02_predictions_and_external_eval.ipynb'
        installer_nb = json.loads(installer_nb_path.read_text(encoding='utf-8'))
        installer_source = None
        for notebook_cell in installer_nb['cells']:
            if notebook_cell.get('cell_type') != 'code':
                continue
            source = ''.join(notebook_cell.get('source', []))
            if 'INSTALL_MODEL_DEPS = True' in source and 'AUTO_PIN_TORCH_FOR_MAMBA' in source:
                installer_source = source
                break
        if installer_source is None:
            raise RuntimeError('Could not locate canonical Mamba installer cell in Notebook 02.')
        exec(compile(installer_source, str(installer_nb_path) + ':model-deps', 'exec'), globals(), globals())
        importlib.invalidate_caches()
    for module_name in ['mamba_ssm', 'causal_conv1d']:
        loaded_module = importlib.import_module(module_name)
        print('OK import:', module_name, getattr(loaded_module, '__file__', 'built-in'))

if RUN_ROBUSTNESS_STRESS and (FORCE_RERUN_ROBUSTNESS_STRESS or not robustness_outputs_ready):
    if not Path('scripts/revision/12_robustness_stress.py').exists():
        raise FileNotFoundError('Missing scripts/revision/12_robustness_stress.py')
    run(robustness_command, log_path='reports/revision/logs/robustness_stress.log')
elif robustness_outputs_ready:
    print('Reusing existing robustness stress artifacts. Set FORCE_RERUN_ROBUSTNESS_STRESS=True to recompute.')
else:
    robustness_rows = []
    for analysis_name, perturbation in [
        ('SNR 20 dB noise', 'snr20db'),
        ('SNR 10 dB noise', 'snr10db'),
        ('SNR 5 dB noise', 'snr5db'),
        ('Random 3-lead dropout', 'random_3_lead_dropout'),
        ('V1-V6 dropout', 'precordial_dropout'),
        ('500Hz to 250Hz sampling shift', 'resample_250hz'),
    ]:
        robustness_rows.append({
            'analysis_name': analysis_name,
            'perturbation': perturbation,
            'dataset': 'chapman_oof',
            'status': 'blocked_not_run',
            'clean_metric_reference': str(calibration_path),
            'perturbed_metric_value': math.nan,
            'absolute_delta': math.nan,
            'relative_delta': math.nan,
            'evidence_path': '',
            'blocker': 'Robustness runner is implemented but RUN_ROBUSTNESS_STRESS=False and no prior artifacts were restored.',
            'safe_wording': 'Do not claim robustness under this perturbation until the runner is executed and artifacts are frozen.',
        })
    robustness_df = pd.DataFrame(robustness_rows)
    robustness_outputs['summary_csv'].parent.mkdir(parents=True, exist_ok=True)
    robustness_outputs['table_csv'].parent.mkdir(parents=True, exist_ok=True)
    robustness_df.to_csv(robustness_outputs['summary_csv'], index=False)
    robustness_df.to_csv(robustness_outputs['table_csv'], index=False)
    print('Wrote blocked robustness ledger:', robustness_outputs['summary_csv'])

for name, path in robustness_outputs.items():
    print(f'{name:18s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

robustness_df = pd.read_csv(robustness_outputs['summary_csv'])
if ROBUSTNESS_FINAL_AGGREGATION_PASS:
    actual_stresses = set(robustness_df['stress_test'].astype(str))
    expected_stresses = set(expected_full_robustness_stresses)
    actual_metrics = set(robustness_df['metric'].astype(str)) if 'metric' in robustness_df.columns else set()
    expected_rows = len(expected_full_robustness_stresses) * len(expected_robustness_metrics)
    if actual_stresses != expected_stresses:
        raise RuntimeError(f'Final robustness summary has wrong stress set: {sorted(actual_stresses)}')
    if actual_metrics != expected_robustness_metrics:
        raise RuntimeError(f'Final robustness summary has wrong metric set: {sorted(actual_metrics)}')
    if len(robustness_df) != expected_rows:
        raise RuntimeError(f'Final robustness summary should have {expected_rows} rows, found {len(robustness_df)}')
    print(f'Final robustness aggregation validated: {len(robustness_df)} rows across {len(expected_full_robustness_stresses)} stresses and {len(expected_robustness_metrics)} metrics.')

display(robustness_df.sort_values(['stress_test', 'metric']).reset_index(drop=True))


## Comparator Stress Prediction Generation

Generate stressed prediction artifacts for ResNet1D/CNN and Raw Mamba from checkpoints saved by Notebook 04. This cell is inference-only and will not retrain. If checkpoints are missing, keep it disabled and rerun Notebook 04 with `*_SAVE_CHECKPOINTS=True` and `*_FORCE_RERUN=True`.


In [ ]:
RUN_COMPARATOR_STRESS_PREDICTIONS = False
COMPARATOR_STRESS_COMPARATORS = 'resnet,raw_mamba'
COMPARATOR_STRESS_TESTS = ROBUSTNESS_STRESS_TESTS
COMPARATOR_STRESS_BATCH_SIZE = 256
COMPARATOR_STRESS_NUM_WORKERS = 0
COMPARATOR_STRESS_STRICT = False

comparator_stress_command = (
    'python -u scripts/revision/23_generate_comparator_stress_predictions.py '
    f'--comparators {COMPARATOR_STRESS_COMPARATORS} '
    f'--stress-tests {COMPARATOR_STRESS_TESTS} '
    '--oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz '
    '--freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json '
    '--expected-checkpoint-kind final_ema '
    f'--batch-size {COMPARATOR_STRESS_BATCH_SIZE} --num-workers {COMPARATOR_STRESS_NUM_WORKERS} '
    '--device cuda --amp --allow-tf32 --reuse-existing'
)
if COMPARATOR_STRESS_STRICT:
    comparator_stress_command += ' --strict'

print('Comparator stress command:', comparator_stress_command)
checkpoint_candidates = [
    *(revision_root / 'experimental' / 'resnet1d_cnn_checkpoints' / f'fold{i}_resnet1d_cnn_final.pt' for i in range(1, 6)),
    *(revision_root / 'experimental' / 'raw_mamba_checkpoints' / f'fold{i}_raw_mamba_final_ema.pt' for i in range(1, 6)),
]
print('Comparator checkpoint status:')
for path in checkpoint_candidates:
    print(f'  {path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')

if RUN_COMPARATOR_STRESS_PREDICTIONS:
    if not Path('scripts/revision/23_generate_comparator_stress_predictions.py').exists():
        raise FileNotFoundError('Missing scripts/revision/23_generate_comparator_stress_predictions.py')
    run(comparator_stress_command, log_path='reports/revision/logs/comparator_stress_predictions.log')
else:
    print('Comparator stress prediction generation disabled. Enable only after Notebook 04 has saved ResNet/Raw Mamba checkpoints.')

for _stress in [item.strip() for item in COMPARATOR_STRESS_TESTS.split(',') if item.strip()]:
    for _stem in ['resnet1d_cnn', 'raw_mamba']:
        path = revision_root / 'predictions' / f'robustness_{_stem}_{_stress}_predictions.npz'
        print(f'{path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
manifest = revision_root / 'manifests' / 'comparator_stress_prediction_manifest.json'
print(f'{manifest}: exists={manifest.exists()} size={manifest.stat().st_size if manifest.exists() else None}')


## Multi-Comparator Robustness Ledger

Optional low-memory aggregation gate for robustness beyond MiniRocket-only. This cell does not generate new stressed predictions. It validates existing clean/stressed artifacts for Full ECG-RAMBA, MiniRocket-only, ResNet1D/CNN, and Raw Mamba, and records missing comparator-stress artifacts as blocked rows.


In [ ]:
# Disabled by default because the first full bootstrap pass can be CPU-bound.
RUN_ROBUSTNESS_MULTICOMPARATOR = False
ROBUSTNESS_MULTI_COMPARATORS = 'full,minirocket,resnet,raw_mamba'
ROBUSTNESS_MULTI_STRESSES = ROBUSTNESS_STRESS_TESTS
ROBUSTNESS_MULTI_N_BOOT = 1000
ROBUSTNESS_MULTI_STRICT = False
ROBUSTNESS_MULTI_METRIC_CACHE_DIR = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision' / 'metrics' / 'robustness_multicomparator_metric_cache'

multi_robustness_command = (
    'python -u scripts/revision/21_robustness_multicomparator.py '
    f'--comparators {ROBUSTNESS_MULTI_COMPARATORS} '
    f'--stress-tests {ROBUSTNESS_MULTI_STRESSES} '
    f'--threshold 0.5 --n-bins 15 --n-boot {ROBUSTNESS_MULTI_N_BOOT} '
    f'--metric-cache-dir "{ROBUSTNESS_MULTI_METRIC_CACHE_DIR}" --reuse-metric-cache'
)
if ROBUSTNESS_MULTI_STRICT:
    multi_robustness_command += ' --strict'

multi_robustness_outputs = {
    'summary': revision_root / 'metrics' / 'robustness_multicomparator_summary.csv',
    'pairwise': revision_root / 'metrics' / 'robustness_multicomparator_pairwise.json',
    'table': revision_root / 'tables' / 'table_robustness_multicomparator.csv',
    'manifest': revision_root / 'manifests' / 'robustness_multicomparator_manifest.json',
}


robustness_multi_required_predictions = []
for _stress in [item.strip() for item in ROBUSTNESS_MULTI_STRESSES.split(',') if item.strip()]:
    for _stem in ['full', 'minirocket', 'resnet1d_cnn', 'raw_mamba']:
        robustness_multi_required_predictions.append(revision_root / 'predictions' / f'robustness_{_stem}_{_stress}_predictions.npz')
robustness_multi_missing_predictions = [path for path in robustness_multi_required_predictions if not path.exists()]
print('Multi-comparator required stress prediction artifacts:')
for path in robustness_multi_required_predictions:
    print(f'  {path}: exists={path.exists()} size={path.stat().st_size if path.exists() else None}')
if robustness_multi_missing_predictions:
    print('Missing stress predictions keep the affected comparator/stress rows blocked. For ResNet/Raw Mamba, first rerun Notebook 04 with *_SAVE_CHECKPOINTS=True and *_FORCE_RERUN=True, then generate stressed predictions from those checkpoints.')

print('Multi-comparator robustness command:', multi_robustness_command)
if RUN_ROBUSTNESS_MULTICOMPARATOR:
    run(multi_robustness_command, log_path='reports/revision/logs/robustness_multicomparator.log')
else:
    print('Multi-comparator robustness disabled. Existing artifacts, if present, remain usable as blocked/completed evidence.')

for label, path in multi_robustness_outputs.items():
    print(f'{label:10s}: exists={path.exists()} size={path.stat().st_size if path.exists() else None} path={path}')

if multi_robustness_outputs['summary'].exists():
    multi_robustness_df = pd.read_csv(multi_robustness_outputs['summary'])
    display(multi_robustness_df.sort_values(['stress_test', 'comparator', 'metric']).reset_index(drop=True))
else:
    print('No multi-comparator robustness ledger present; broad robustness remains unsupported.')


## Claim Evidence Summary

This JSON is the rebuttal-facing interpretation boundary for HRV and robustness claims.


In [ ]:
hrv_summary_csv = revision_root / 'metrics' / 'hrv_domain_summary.csv'
hrv_df = pd.read_csv(hrv_summary_csv)
hrv_statuses = dict(zip(hrv_df['analysis_name'], hrv_df['status']))
hrv_baseline_summary_path = revision_root / 'metrics' / 'hrv_only_baseline_summary.json'
hrv_domain_classifier_summary_path = revision_root / 'metrics' / 'hrv_domain_classifier_summary.json'
hrv_baseline_summary = json.loads(hrv_baseline_summary_path.read_text(encoding='utf-8'))
hrv_domain_classifier_summary = json.loads(hrv_domain_classifier_summary_path.read_text(encoding='utf-8'))
hrv_only_complete = hrv_statuses.get('HRV-only baseline') == 'complete'
hrv_domain_complete = hrv_statuses.get('HRV domain classifier') == 'complete'
robustness_comparison_path = revision_root / 'metrics' / 'robustness_full_vs_minirocket_comparison.json'
robustness_summary_path = revision_root / 'metrics' / 'robustness_summary.csv'
robustness_payload = None
robustness_complete = False
if robustness_comparison_path.exists():
    robustness_payload = json.loads(robustness_comparison_path.read_text(encoding='utf-8'))
    robustness_complete = robustness_payload.get('status') is True
if hrv_only_complete and hrv_domain_complete:
    hrv_status = 'hrv_only_and_domain_classifier_complete_duration_noise_blocked'
elif hrv_only_complete:
    hrv_status = 'hrv_only_complete_domain_classifier_blocked_or_missing_duration_noise_blocked'
else:
    hrv_status = 'blocked_or_incomplete'

summary = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': git_commit(),
    'source_freeze_manifest': str(freeze_path),
    'source_calibration': str(calibration_path),
    'hrv_schema': {'path': str(hrv_schema_path), 'sha256': sha256_file(hrv_schema_path)},
    'hrv_status': hrv_status,
    'robustness_status': 'complete_perturbation_stress_tests' if robustness_complete else 'blocked_or_not_run',
    'robustness_artifacts': {
        'comparison_json': str(robustness_comparison_path) if robustness_comparison_path.exists() else None,
        'summary_csv': str(robustness_summary_path) if robustness_summary_path.exists() else None,
        'status': robustness_payload.get('status') if robustness_payload else False,
    },
    'hrv_metrics': {
        'hrv_only_roc_auc_macro': hrv_baseline_summary['metrics'].get('roc_auc_macro'),
        'hrv_only_pr_auc_macro': hrv_baseline_summary['metrics'].get('pr_auc_macro'),
        'hrv_only_f1_macro': hrv_baseline_summary['metrics'].get('f1_macro'),
        'domain_status': hrv_domain_classifier_summary.get('status', 'complete' if hrv_domain_complete else 'blocked_or_missing'),
        'domain_roc_auc_ovr_macro': (hrv_domain_classifier_summary.get('metrics') or {}).get('domain_roc_auc_ovr_macro'),
        'domain_balanced_accuracy': (hrv_domain_classifier_summary.get('metrics') or {}).get('balanced_accuracy'),
        'domain_interpretation': hrv_domain_classifier_summary.get('interpretation'),
        'domain_blocker': hrv_domain_classifier_summary.get('blocker'),
    },
    'claim_boundaries': [
        {
            'claim_id': 'C03',
            'claim_topic': 'HRV provides complementary rhythm descriptors',
            'current_status': 'partially_supported_by_hrv_only_only' if hrv_only_complete and not hrv_domain_complete else ('partially_supported_by_hrv_only_and_domain_classifier' if hrv_only_complete else 'not_supported_by_current_artifacts'),
            'safe_wording': 'HRV-only can be reported as a separate feature baseline. Domain-sensitivity evidence requires external HRV36 caches if the domain classifier is blocked. Duration/noise HRV sensitivity remains blocked.',
        },
        {
            'claim_id': 'A6/R2C3',
            'claim_topic': 'minimum robustness stress tests',
            'current_status': 'supported_by_perturbation_stress_artifacts' if robustness_complete else 'not_supported_by_current_artifacts',
            'safe_wording': 'Use only metric- and perturbation-specific robustness claims where paired degradation CIs favor Full ECG-RAMBA.' if robustness_complete else 'Robustness tests must be run and reported as absolute/relative degradation before any robustness claim.',
        },
    ],
}
summary_path = revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json'
save_json(summary_path, summary)
print('Wrote:', summary_path)
print(json.dumps(summary['hrv_metrics'], indent=2, sort_keys=True))


## Remaining Runner Guard

This cell records implemented runners and keeps any non-implemented follow-up work explicit.


In [ ]:
planned = {
    'HRV-only and domain analysis': 'python scripts/revision/09_hrv_domain_analysis.py --oof-predictions reports/revision/predictions/oof_final_ema_predictions.npz --freeze-manifest reports/revision/manifests/oof_final_ema_freeze_manifest.json --expected-checkpoint-kind final_ema --chapman-hrv-cache /content/drive/MyDrive/ECG-Ramba/hrv36_N44186_C12_L5000_R{dataset_record_order_fingerprint}.npz',
    'Full vs MiniRocket robustness stress tests': 'python scripts/revision/12_robustness_stress.py --stress-tests snr20db,snr10db,snr5db,random_3_lead_dropout,precordial_dropout,resample_250hz',
    'Duration/noise HRV sensitivity': 'python scripts/revision/TBD_hrv_duration_noise_sensitivity.py',
}

for name, command in planned.items():
    script = Path(command.split()[1])
    print(f'{name:36s}: implemented={script.exists()} planned_command={command}')

if RUN_ROBUSTNESS_STRESS:
    stress_script = Path('scripts/revision/12_robustness_stress.py')
    if not stress_script.exists():
        raise RuntimeError('Robustness stress runner is missing: ' + str(stress_script))
else:
    print('Robustness stress execution disabled in this run. Existing artifacts, if restored, remain usable; otherwise robustness claims stay blocked.')


## Inventory And Stable Mirror

Freeze the revised artifact inventory and publish all reusable outputs to the Drive mirror.


In [ ]:
import inspect
if 'log_path' not in inspect.signature(run).parameters:
    raise RuntimeError('Notebook command helper run() was overwritten. Rerun the Setup cell before this inventory/mirror cell.')

run(
    'python -u scripts/revision/05_artifact_inventory.py',
    log_path='reports/revision/logs/hrv_robustness_artifact_inventory.log',
)
mirror_root = DRIVE_ROOT / 'revision_artifacts' / 'reports' / 'revision'
run(
    f'python -u scripts/revision/artifact_mirror.py publish --mirror-root "{mirror_root}"',
    log_path='reports/revision/logs/hrv_robustness_mirror_publish.log',
)


## Output Summary


In [ ]:
expected_outputs = [
    revision_root / 'manifests' / 'hrv_robustness_input_contract.json',
    revision_root / 'manifests' / 'hrv_robustness_plan_alignment.json',
    revision_root / 'manifests' / 'hrv_domain_analysis_manifest.json',
    revision_root / 'predictions' / 'hrv_only_oof_predictions.npz',
    revision_root / 'metrics' / 'hrv_only_baseline_summary.json',
    revision_root / 'metrics' / 'hrv_domain_classifier_summary.json',
    revision_root / 'metrics' / 'hrv_domain_summary.csv',
    revision_root / 'tables' / 'table_hrv_domain_status.csv',
    revision_root / 'tables' / 'table_hrv_only_class_metrics.csv',
    revision_root / 'metrics' / 'robustness_summary.csv',
    revision_root / 'tables' / 'table_robustness.csv',
    revision_root / 'metrics' / 'hrv_robustness_component_check_summary.json',
    revision_root / 'manifests' / 'artifacts_manifest.json',
    revision_root / 'manifests' / 'artifacts_manifest.csv',
]
optional_outputs = [
    revision_root / 'predictions' / 'hrv_domain_oof_predictions.npz',
    revision_root / 'tables' / 'table_hrv_domain_classifier_confusion.csv',
    revision_root / 'tables' / 'table_hrv_domain_classifier_fold_summary.csv',
    revision_root / 'metrics' / 'robustness_full_vs_minirocket_comparison.json',
    revision_root / 'metrics' / 'robustness_full_vs_minirocket_bootstrap_samples.csv',
    revision_root / 'manifests' / 'robustness_stress_manifest.json',
    revision_root / 'tables' / 'table_robustness_stress_metadata.csv',
    revision_root / 'tables' / 'table_robustness_artifacts.csv',
]
robustness_reference_outputs = [
    revision_root / 'predictions' / 'robustness_minirocket_clean_ref_predictions.npz',
    revision_root / 'manifests' / 'robustness_minirocket_heads_manifest.json',
]
robustness_prediction_outputs = []
for stress_name in [
    'snr20db',
    'snr10db',
    'snr5db',
    'random_3_lead_dropout',
    'precordial_dropout',
    'resample_250hz',
]:
    robustness_prediction_outputs.extend([
        revision_root / 'predictions' / f'robustness_full_{stress_name}_predictions.npz',
        revision_root / 'predictions' / f'robustness_minirocket_{stress_name}_predictions.npz',
    ])

for path in expected_outputs:
    print(path, 'required exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)
for path in optional_outputs:
    print(path, 'optional exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)
for path in robustness_reference_outputs:
    print(path, 'robustness reference exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)
for path in robustness_prediction_outputs:
    print(path, 'robustness prediction exists=', path.exists(), 'size=', path.stat().st_size if path.exists() else None)

print()
missing_required = [str(path) for path in expected_outputs if not path.exists()]
missing_references = [str(path) for path in robustness_reference_outputs if not path.exists()]
missing_predictions = [str(path) for path in robustness_prediction_outputs if not path.exists()]
if missing_required:
    raise FileNotFoundError('Notebook 05 required outputs are missing: ' + '; '.join(missing_required))
if missing_references:
    raise FileNotFoundError('Notebook 05 robustness reference outputs are missing: ' + '; '.join(missing_references))
if missing_predictions:
    raise FileNotFoundError('Notebook 05 robustness prediction outputs are missing: ' + '; '.join(missing_predictions))

robustness_summary_path = revision_root / 'metrics' / 'robustness_summary.csv'
robustness_manifest_path = revision_root / 'manifests' / 'robustness_stress_manifest.json'
if robustness_summary_path.exists() and robustness_manifest_path.exists():
    robustness_summary = pd.read_csv(robustness_summary_path)
    expected_stresses = {
        'snr20db',
        'snr10db',
        'snr5db',
        'random_3_lead_dropout',
        'precordial_dropout',
        'resample_250hz',
    }
    expected_metrics = {'pr_auc_macro', 'roc_auc_macro', 'f1_macro', 'brier_macro', 'ece_macro'}
    actual_stresses = set(robustness_summary['stress_test'].astype(str))
    actual_metrics = set(robustness_summary['metric'].astype(str))
    if actual_stresses != expected_stresses:
        raise RuntimeError(f'Robustness summary stress set is incomplete: {sorted(actual_stresses)}')
    if actual_metrics != expected_metrics:
        raise RuntimeError(f'Robustness summary metric set is incomplete: {sorted(actual_metrics)}')
    if len(robustness_summary) != len(expected_stresses) * len(expected_metrics):
        raise RuntimeError(f'Robustness summary row count is incomplete: {len(robustness_summary)}')
    manifest_payload = json.loads(robustness_manifest_path.read_text(encoding='utf-8'))
    manifest_stresses = set(manifest_payload.get('args', {}).get('stress_tests', '').split(','))
    if manifest_stresses != expected_stresses:
        raise RuntimeError(f'Robustness manifest stress set is incomplete: {sorted(manifest_stresses)}')
    print('Notebook 05 robustness final summary validated: full six-stress table is present and manifest matches.')
    display(robustness_summary.sort_values(['stress_test', 'metric']).reset_index(drop=True))

if (revision_root / 'metrics' / 'robustness_full_vs_minirocket_comparison.json').exists():
    print('Notebook 05 complete. HRV/domain evidence and perturbation robustness artifacts are present. Use only metric-specific robustness claims supported by paired degradation CIs.')
else:
    print('Notebook 05 complete. HRV-only evidence is implemented; HRV domain evidence is complete only when external HRV36 caches exist. Robustness claims remain blocked until perturbation inference artifacts exist.')
